# Level 1 - The glomerulus: a filter driven by pressure
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/kidney-sim/blob/main/notebooks/01_glomerulus_pressure.ipynb)

**Goal:** compute the GFR (glomerular filtration rate) from physics: pressures and flows.

**Answer to the question "does pressure matter in the body?":** in the glomerulus, pressure is *everything*.

In [ ]:
# --- Setup: run this cell first --------------------------------------------
import os, sys, subprocess
REPO_URL = "https://github.com/YOUR-USERNAME/kidney-sim.git"   # change if you forked the project
if "google.colab" in sys.modules and not os.path.exists("kidneysim"):
    if not os.path.exists("kidney-sim"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("kidney-sim")
for path in [".", ".."]:
    if os.path.exists(os.path.join(path, "kidneysim")):
        sys.path.insert(0, os.path.abspath(path))
        break

import numpy as np
import matplotlib.pyplot as plt
import kidneysim as ks
from kidneysim import viz2d, viz3d
from kidneysim.ui import sliders
print("kidneysim", ks.__version__, "| interactive 3D (plotly):", viz3d.HAS_PLOTLY)

## 1. Blood flow through the kidney = an electric circuit

Blood enters through the **afferent** arteriole, crosses the glomerulus and leaves through the **efferent** arteriole.
Each vessel is a resistance:

$$ Q_{blood} = \frac{MAP - P_{vein}}{R_a + R_e} \qquad\qquad P_{GC} = MAP - Q_{blood}\,R_a $$

The efferent vessel is narrow, so it keeps the pressure inside the glomerulus high (~60 mmHg).

## 2. Starling equation along the capillary

$$ \frac{dQ_f}{dx} = K_f\Big[\underbrace{(P_{GC}-P_{BS})}_{\text{pushes out}} - \underbrace{(\pi_{GC}-\pi_{BS})}_{\text{pulls back}}\Big] $$

* $P_{GC}$ blood pressure in the capillary, $P_{BS}$ pressure in Bowman's space (15 mmHg)
* $\pi_{GC}$ **oncotic pressure** of proteins. As water leaves, proteins get concentrated:

$$ c(x) = c_0\,\frac{Q_p(0)}{Q_p(x)}, \qquad \pi = 2.1c + 0.16c^2 + 0.009c^3 \;\;(\text{Landis–Pappenheimer}) $$

In [ ]:
f = ks.glomerulus.filtration_along_capillary()
print(f"GFR = {f['gfr']*1000:.0f} mL/min  ({f['gfr']*1440:.0f} L/day!)")
print(f"filtration fraction = {100*f['filtration_fraction']:.0f} % of the plasma")
viz2d.plot_capillary(f);

### 3D: the capillary ball, coloured by net filtration pressure
Rotate it with the mouse (in Colab). Red = strong filtration, blue = almost none.

In [ ]:
viz3d.show(viz3d.glomerulus_3d(f))

## 3. Autoregulation

If blood pressure goes up, the afferent arteriole contracts (myogenic response) so the flow stays the same:

$$ R_a = \frac{MAP-P_{vein}}{Q_{target}} - R_e \qquad (\text{limited to } 0.5\text{–}3\times \text{normal}) $$

In [ ]:
maps = np.linspace(40, 200, 81)
auto = [ks.glomerulus.gfr_from_map(m, autoregulation=True) for m in maps]
rigid = [ks.glomerulus.gfr_from_map(m, autoregulation=False) for m in maps]
viz2d.plot_autoregulation(maps, auto, rigid);

### 3D: GFR versus blood pressure *and* filter health
$K_f$ falls when glomeruli are damaged (diabetes, high blood pressure over years).

In [ ]:
viz3d.show(viz3d.gfr_surface())

## 4. What passes the filter? (sieving)

$$ \lambda = \frac{r_{molecule}}{r_{pore}},\qquad \theta = (1-\lambda)^2\,\big(2-(1-\lambda)^2\big) $$

Albumin is also **negatively charged** and so is the filter: it is pushed away (extra penalty).

In [ ]:
radii = np.linspace(0, 4.5, 200)
mols = {name: (r, ks.glomerulus.sieving_coefficient(r, charged=c))
        for name, (r, c) in ks.glomerulus.MOLECULES.items()}
viz2d.plot_sieving(radii, ks.glomerulus.sieving_coefficient(radii), mols);

In [ ]:
def experiment(MAP=100.0, efferent_resistance=1.0, filter_health=1.0):
    h = ks.glomerulus.hemodynamics(MAP, R_e=ks.glomerulus.R_E_NORMAL * efferent_resistance,
                                   autoregulation=True)
    f = ks.glomerulus.filtration_along_capillary(P_gc=h["P_gc"], Q_plasma=h["Q_plasma"],
                                                 Kf=ks.glomerulus.KF_NORMAL * filter_health)
    viz2d.plot_capillary(f)
    plt.show()

sliders(experiment, MAP=(40, 200, 5, 100), efferent_resistance=(0.5, 2.0, 0.1, 1.0),
        filter_health=(0.1, 1.0, 0.05, 1.0))

## Questions
1. What happens to GFR during a severe haemorrhage (MAP 50 mmHg)? Why is that dangerous for the kidney?
2. Squeezing the **efferent** arteriole (angiotensin II does this) changes $P_{GC}$. Does GFR go up or down?
3. Why is protein in the urine (albuminuria) a sign of a damaged filter?

## What is missing?
180 L/day are filtered but we only pee ~1.5 L. Where does the rest go? → Level 2.